In [1]:
import pandas as pd
from sqlalchemy import create_engine

# Connect to the database
db_url = "sqlite:///stores/rpdr.db"
engine = create_engine(db_url)

# Query the Phy table for first 3 records
query = "SELECT * FROM phy LIMIT 3"
phy_df = pd.read_sql(query, engine)

def is_smoker_vectorized(phy_subset):
    """Vectorized smoking detection - same logic as make_cohort.py"""
    concept_name = phy_subset["Concept_Name"]
    result_lower = phy_subset["Result"].astype(str).str.lower()

    # Initialize all as False
    is_smoker_flag = pd.Series(False, index=phy_subset.index)

    # Smoking status checks
    is_smoker_flag |= (
        (concept_name == "Smoking status") &
        result_lower.isin([
            "current every day smoker", "smoker, current status unknown",
            "former smoker", "quit tobacco >= 1 year ago",
            "current tobacco user", "1/2 ppd"
        ])
    )

    # Concept name checks
    is_smoker_flag |= concept_name.isin([
        "Smoking Tobacco Use-Former Smoker",
        "Smoking Tobacco Use-Current Every Day Smoker",
        "Smoking Tobacco Use-Current Some Day Smoker",
        "Smoking Tobacco Use-Light Tobacco Smoker",
        "Smoking Tobacco Use-Smoker, Current Status Unknown",
        "Smoking Tobacco Use-Heavy Tobacco Smoker",
        "Tobacco User-Yes", "Tobacco User-Quit",
        "Cigarettes", "Tobacco Pack Per Day"
    ])

    # Smoker yes/true checks
    is_smoker_flag |= (concept_name == "Smoker") & result_lower.isin(["yes", "1", "true", "1.0"])

    # Tobacco years used check
    is_smoker_flag |= (concept_name == "Tobacco Used Years") & ~result_lower.isin(["0", "0.0", ""])

    return is_smoker_flag

# Apply smoking detection
smoking_flags = is_smoker_vectorized(phy_df)

# Print results
print("First 3 records from Phy table:")
for idx, row in phy_df.iterrows():
    is_smoker = smoking_flags[idx]
    status = "SMOKER" if is_smoker else "NOT SMOKER"
    print(f"\n{status}:")
    print(f"  Concept_Name: {row['Concept_Name']}")
    print(f"  Result: {row['Result']}")

First 3 records from Phy table:

NOT SMOKER:
  Concept_Name: BMI
  Result: 25.51

NOT SMOKER:
  Concept_Name: BMI
  Result: 26.93

NOT SMOKER:
  Concept_Name: BMI
  Result: 27.22


In [2]:
import pandas as pd
from sqlalchemy import create_engine

# Connect to the database
db_url = "sqlite:///stores/rpdr.db"
engine = create_engine(db_url)

# Query the Phy table for first 3 records
query = "SELECT DISTINCT Concept_Name FROM phy"
phy_df = pd.read_sql(query, engine)
# print(phy_df.Code.unique())
print(phy_df)

                                          Concept_Name
0                                                  BMI
1                                  Blood Pressure-Epic
2                              Body Surface Area (BSA)
3                                       Diastolic-Epic
4                                               Height
..                                                 ...
555                          Meningococcal C/Y-HIB PRP
556                                       Dengue Fever
557               Inactivated Poliovirus vaccine (IPV)
558  Diphtheria,Tetanus, accellular Pertussis vacci...
559                           O2 Saturation-LFA15000.1

[560 rows x 1 columns]


In [3]:
# Step 1: Get all alcohol-related concept names
alcohol_concepts = phy_df[phy_df['Concept_Name'].str.contains('alcohol', case=False, na=False)]
print(f"Found {len(alcohol_concepts)} alcohol-related concept names:\n")
print(alcohol_concepts)

Found 13 alcohol-related concept names:

                                          Concept_Name
10                                     Alcohol User-No
45                                    Alcohol User-Yes
46                                 Alcohol Oz Per Week
47                                 Alcohol Type - Wine
48                                 Alcohol Type - Beer
49   Alcohol Type - Drinks containing 0.5 oz of alc...
50                              Alcohol Type - Unknown
51                             Alcohol Drinks Per Week
98                          Alcohol User-Not Currently
114                              Alcohol Use Screening
132                     Alcohol Type - Shots of liquor
135                             Alcohol User-Not Asked
189                                 Alcohol User-Never


In [4]:
# Step 2: For each alcohol concept, get the possible Result values and their counts
alcohol_concept_names = alcohol_concepts['Concept_Name'].tolist()

print("Analyzing Results for each alcohol-related Concept_Name:\n")
print("="*80)

# Build query with properly quoted concept names
quoted_names = ','.join([f"'{name}'" for name in alcohol_concept_names])
alcohol_query = f"""
SELECT Concept_Name, Result, COUNT(*) as count 
FROM phy 
WHERE Concept_Name IN ({quoted_names})
GROUP BY Concept_Name, Result
ORDER BY Concept_Name, count DESC
"""

alcohol_results = pd.read_sql(alcohol_query, engine)
print(alcohol_results)

Analyzing Results for each alcohol-related Concept_Name:

                    Concept_Name Result   count
0        Alcohol Drinks Per Week      0  209163
1        Alcohol Drinks Per Week      1   85727
2        Alcohol Drinks Per Week      2   58670
3        Alcohol Drinks Per Week      3   30153
4        Alcohol Drinks Per Week      4   24192
...                          ...    ...     ...
3565          Alcohol User-Never          28265
3566             Alcohol User-No         384631
3567      Alcohol User-Not Asked          23809
3568  Alcohol User-Not Currently          99586
3569            Alcohol User-Yes         601958

[3570 rows x 3 columns]


In [5]:
# Step 3: Filter for results with >10 hits and classify them
print("\n\nResults with >10 hits - CLASSIFICATION:\n")
print("="*80)

# Filter for >10 hits
significant_results = alcohol_results[alcohol_results['count'] > 10].copy()

# Function to classify alcohol status based on concept name and result
def classify_alcohol_status(row):
    """
    Classify as:
    - CURRENT: Currently drinks alcohol
    - FORMER: Previously drank but not currently
    - NEVER: Never drank alcohol
    - UNKNOWN: Cannot determine from this field alone
    """
    concept = row['Concept_Name']
    result = str(row['Result']).lower()
    
    # Current drinker indicators
    if concept == "Alcohol User-Yes":
        return "CURRENT"
    elif concept in ["Alcohol Type - Beer", "Alcohol Type - Shots of liquor", 
                     "Alcohol Type - Wine", "Alcohol Type - Unknown",
                     "Alcohol Type - Drinks containing 0.5 oz of alcohol"]:
        return "CURRENT (by presence)"
    elif concept in ["Alcohol Drinks Per Week", "Alcohol Oz Per Week"]:
        if result not in ["0", "0.0", "", "none", "nan"]:
            return "CURRENT"
        elif result in ["0", "0.0"]:
            return "NOT CURRENT (0 drinks)"
        else:
            return "UNKNOWN"
    
    # Not currently drinking indicators
    elif concept in ["Alcohol User-No", "Alcohol User-Never", "Alcohol User-Not Currently"]:
        if "never" in concept.lower():
            return "NEVER"
        elif "not currently" in concept.lower():
            return "NOT CURRENT (former or never)"
        else:
            return "NOT CURRENT"
    
    # Screening/assessment (doesn't indicate status)
    elif "screening" in concept.lower() or "not asked" in concept.lower():
        return "SCREENING/NOT ASKED"
    
    return "UNKNOWN"

# Apply classification
significant_results['classification'] = significant_results.apply(classify_alcohol_status, axis=1)

# Display results sorted by concept and count
for concept in significant_results['Concept_Name'].unique():
    subset = significant_results[significant_results['Concept_Name'] == concept]
    print(f"\n{concept}:")
    for _, row in subset.iterrows():
        print(f"  Result: '{row['Result']}' (n={row['count']:,}) → {row['classification']}")



Results with >10 hits - CLASSIFICATION:


Alcohol Drinks Per Week:
  Result: '0' (n=209,163) → NOT CURRENT (0 drinks)
  Result: '1' (n=85,727) → CURRENT
  Result: '2' (n=58,670) → CURRENT
  Result: '3' (n=30,153) → CURRENT
  Result: '4' (n=24,192) → CURRENT
  Result: '7' (n=18,542) → CURRENT
  Result: '0-1' (n=18,369) → CURRENT
  Result: '5' (n=17,251) → CURRENT
  Result: '1-2' (n=16,796) → CURRENT
  Result: '6' (n=11,055) → CURRENT
  Result: '2-3' (n=10,369) → CURRENT
  Result: '0-2' (n=9,978) → CURRENT
  Result: '10' (n=6,984) → CURRENT
  Result: '14' (n=6,537) → CURRENT
  Result: '3-4' (n=6,353) → CURRENT
  Result: '8' (n=3,410) → CURRENT
  Result: '4-5' (n=3,220) → CURRENT
  Result: '3-6' (n=2,931) → CURRENT
  Result: '0-3' (n=2,915) → CURRENT
  Result: '2-4' (n=2,750) → CURRENT
  Result: '12' (n=2,020) → CURRENT
  Result: '3-5' (n=2,009) → CURRENT
  Result: '4-6' (n=1,504) → CURRENT
  Result: '0-4' (n=1,483) → CURRENT
  Result: '5-6' (n=1,349) → CURRENT
  Result: '7-10' (n=1,272

In [6]:
# Step 4: Summary - Create a lookup table for structured data extraction
print("\n\n" + "="*80)
print("SUMMARY: Alcohol Status Classification Guide")
print("="*80)

# Group by classification
summary = significant_results.groupby('classification').agg({
    'Concept_Name': lambda x: list(set(x)),
    'count': 'sum'
}).reset_index()

print("\nBy Classification:")
for _, row in summary.iterrows():
    print(f"\n{row['classification']} (total records: {row['count']:,}):")
    for concept in row['Concept_Name']:
        concept_results = significant_results[
            (significant_results['Concept_Name'] == concept) & 
            (significant_results['classification'] == row['classification'])
        ]
        for _, r in concept_results.iterrows():
            print(f"  - {concept}: '{r['Result']}' (n={r['count']:,})")



SUMMARY: Alcohol Status Classification Guide

By Classification:

CURRENT (total records: 1,373,496):
  - Alcohol User-Yes: '' (n=601,958)
  - Alcohol Oz Per Week: '0.6' (n=82,045)
  - Alcohol Oz Per Week: '1.2' (n=63,577)
  - Alcohol Oz Per Week: '1.8' (n=33,391)
  - Alcohol Oz Per Week: '2.4' (n=28,446)
  - Alcohol Oz Per Week: '4.2' (n=20,371)
  - Alcohol Oz Per Week: '3' (n=19,752)
  - Alcohol Oz Per Week: '0 - .6' (n=15,587)
  - Alcohol Oz Per Week: '.6 - 1.2' (n=14,683)
  - Alcohol Oz Per Week: '3.6' (n=13,585)
  - Alcohol Oz Per Week: '0 - 1.2' (n=9,590)
  - Alcohol Oz Per Week: '1.2 - 1.8' (n=9,384)
  - Alcohol Oz Per Week: '6' (n=7,976)
  - Alcohol Oz Per Week: '8.4' (n=7,377)
  - Alcohol Oz Per Week: '1.8 - 2.4' (n=5,798)
  - Alcohol Oz Per Week: '4.8' (n=4,767)
  - Alcohol Oz Per Week: '2.4 - 3' (n=3,188)
  - Alcohol Oz Per Week: '1.8 - 3.6' (n=2,954)
  - Alcohol Oz Per Week: '0 - 1.8' (n=2,844)
  - Alcohol Oz Per Week: '1.2 - 2.4' (n=2,750)
  - Alcohol Oz Per Week: '7.2' 

# Smoking Status Analysis

Now let's analyze smoking-related concept names the same way we did for alcohol.

In [ ]:
# Step 1: Get all smoking/tobacco-related concept names
smoking_concepts = phy_df[
    phy_df['Concept_Name'].str.contains('smoking|tobacco|smoker', case=False, na=False)
]
print(f"Found {len(smoking_concepts)} smoking/tobacco-related concept names:\n")
print(smoking_concepts)

In [ ]:
# Step 2: For each smoking concept, get the possible Result values and their counts
smoking_concept_names = smoking_concepts['Concept_Name'].tolist()

print("Analyzing Results for each smoking-related Concept_Name:\n")
print("="*80)

# Build query with properly quoted concept names
quoted_names = ','.join([f"'{name}'" for name in smoking_concept_names])
smoking_query = f"""
SELECT Concept_Name, Result, COUNT(*) as count 
FROM phy 
WHERE Concept_Name IN ({quoted_names})
GROUP BY Concept_Name, Result
ORDER BY Concept_Name, count DESC
"""

smoking_results = pd.read_sql(smoking_query, engine)
print(smoking_results)

In [ ]:
# Step 3: Filter for results with >10 hits and classify them
print("\n\nResults with >10 hits - CLASSIFICATION:\n")
print("="*80)

# Filter for >10 hits
significant_smoking = smoking_results[smoking_results['count'] > 10].copy()

# Function to classify smoking status based on concept name and result
def classify_smoking_status(row):
    """
    Classify as:
    - NEVER: Never smoked
    - FORMER: Former smoker
    - CURRENT: Current smoker
    - UNKNOWN: Cannot determine from this field alone
    """
    concept = row['Concept_Name']
    result = str(row['Result']).lower().strip()
    
    # Check Smoking status field with standardized values
    if concept == "Smoking status":
        if result in ["never smoker", "never smoked"]:
            return "NEVER"
        elif result in ["former smoker", "quit tobacco >= 1 year ago"]:
            return "FORMER"
        elif result in [
            "current every day smoker",
            "current some day smoker",
            "smoker, current status unknown",
            "current tobacco user",
        ]:
            return "CURRENT"
        else:
            return f"UNKNOWN (status: {result})"
    
    # Check Smoking Tobacco Use concept names
    elif concept == "Smoking Tobacco Use-Never Smoker":
        return "NEVER"
    elif concept == "Smoking Tobacco Use-Former Smoker":
        return "FORMER"
    elif concept in [
        "Smoking Tobacco Use-Current Every Day Smoker",
        "Smoking Tobacco Use-Current Some Day Smoker",
        "Smoking Tobacco Use-Smoker, Current Status Unknown",
        "Smoking Tobacco Use-Heavy Tobacco Smoker",
        "Smoking Tobacco Use-Light Tobacco Smoker",
    ]:
        return "CURRENT"
    
    # Check Tobacco User fields
    elif concept == "Tobacco User-Never":
        return "NEVER"
    elif concept == "Tobacco User-Quit":
        return "FORMER"
    elif concept == "Tobacco User-Yes":
        return "CURRENT"
    
    # Check Smoker field (depends on result)
    elif concept == "Smoker":
        if result in ["yes", "1", "true", "1.0"]:
            return "CURRENT"
        elif result in ["no", "0", "false", "0.0"]:
            return "NEVER/UNKNOWN"
        else:
            return f"UNKNOWN (smoker: {result})"
    
    # Check quantity fields
    elif concept in ["Cigarettes", "Tobacco Pack Per Day"]:
        return "CURRENT (by presence)"
    
    elif concept == "Tobacco Used Years":
        if result not in ["0", "0.0", ""]:
            return "CURRENT/FORMER (used tobacco)"
        else:
            return "NEVER (0 years)"
    
    # Screening/assessment
    elif "screening" in concept.lower() or "not asked" in concept.lower():
        return "SCREENING/NOT ASKED"
    
    return "UNKNOWN"

# Apply classification
significant_smoking['classification'] = significant_smoking.apply(classify_smoking_status, axis=1)

# Display results sorted by concept and count
for concept in sorted(significant_smoking['Concept_Name'].unique()):
    subset = significant_smoking[significant_smoking['Concept_Name'] == concept]
    total = subset['count'].sum()
    print(f"\n{concept} (total: {total:,} records):")
    for _, row in subset.iterrows():
        print(f"  '{row['Result']}' (n={row['count']:,}) → {row['classification']}")

In [ ]:
# Step 4: Summary - Create a lookup table for structured data extraction
print("\n\n" + "="*80)
print("SUMMARY: Smoking Status Classification Guide")
print("="*80)

# Group by classification
smoking_summary = significant_smoking.groupby('classification').agg({
    'Concept_Name': lambda x: sorted(list(set(x))),
    'count': 'sum'
}).reset_index()

print("\nBy Classification:")
for _, row in smoking_summary.iterrows():
    print(f"\n{row['classification']} (total: {row['count']:,} records)")
    for concept in row['Concept_Name']:
        print(f"  - {concept}")